# COMP 7712 — Exam Summary & Study Guide

**Final Exam Topics:** Running Time Equations · Big-O/Ω/Θ · Master Theorem · Explaining Strategies in English · Divide & Conquer · Dynamic Programming · Backtracking

---

This notebook consolidates the essence of all lecture notes (01–16) and all 7 assignment solutions.
Each section includes the **exact proof/solution technique** used in assignments.

---

# Section 1 — Running Time Equations T(n)

## How to write T(n)

T(n) = the total number of steps a program takes on input of size n.

### Rules for counting steps
- Single assignment / comparison / arithmetic → **constant** (use letters: a, b, c)
- `for x in L:` (length n) → **n iterations**
- Slicing `L[a:b]` → costs proportional to the **slice length** (not free!)
- Function call `f(L[1:])` on a list of size n → `T(n-1)` (because `L[1:]` has n-1 items)
- Function call `f(L[:n//2])` → `T(n/2)`

## 1A. Iterative Programs

### Single loop
```python
total = 0               # constant: a
for i in range(n):      # n times
    total += i          #   constant: b
```
$$T(n) = a + b \cdot n \in \Theta(n)$$

---

### Nested loop — full square
```python
total = 0               # constant: a
for i in range(n):      # n times
    for j in range(n):  #   n times
        total += i*j    #     constant: b
```
$$T(n) = a + b \cdot n^2 \in \Theta(n^2)$$

---

### Nested loop — triangular (inner starts at i+1)
```python
total = 0                  # constant: a
for i in range(n):         # n times
    for j in range(i+1,n): #   runs (n-1), (n-2), ..., 0 times
        total += L[i]*L[j] #     constant: b
```
Inner loop total = $(n-1) + (n-2) + \cdots + 0 = \frac{n(n-1)}{2}$

$$T(n) = a + b \cdot \frac{n(n-1)}{2} \in \Theta(n^2)$$

---

### Key trick: the inversion count pattern
Any time you see `for i in range(n): for j in range(i+1, n):`, the total iterations are $\frac{n(n-1)}{2}$, which is $\Theta(n^2)$.

In [1]:
# WORKED EXAMPLE — count inversions (Assignment 3, Q5)
def has_duplicates(L):
    n = len(L)
    for i in range(n):           # n times
        for j in range(i+1, n):  # (n-1)+(n-2)+...+0 = n(n-1)/2 times total
            if L[i] == L[j]:     # constant
                return True
    return False
# T(n) = a + b * n(n-1)/2  →  Θ(n²)

print(has_duplicates([1,2,3,4]))   # False
print(has_duplicates([1,2,1,4]))   # True

False
True


## 1B. Recursive Programs

Write T(n) by counting the work at the current level (excluding recursive calls), then add T(smaller input).

### Key pattern table

| Program Shape | T(n) recurrence | Complexity |
|---|---|---|
| `if L==[]: ... ; return f(L[1:])` | $T(n) = a + T(n-1)$ | $\Theta(n)$ |
| `remaining = L[1:]; return L[0] + f(remaining)` | $T(n) = n + T(n-1)$ | $\Theta(n^2)$ |
| `left = L[:n//2]; return f(left)` (one recursive call, half input) | $T(n) = n + T(n/2)$ | $\Theta(n)$ |
| `left=L[:n//2]; right=L[n//2:]; return f(left) + f(right)` (merge step = n) | $T(n) = n + 2T(n/2)$ | $\Theta(n \log n)$ |
| `return f(L[:n//2]) + f(L[n//2:])` (combine = O(1)) | $T(n) = n + 2T(n/2)$ | $\Theta(n \log n)$ |
| `return 1 + f(L[1:])` (no slicing, just index) | $T(n) = a + T(n-1)$ | $\Theta(n)$ |

**Important:** Slicing `L[1:]` on a list of size n costs **n steps** (copies n-1 items). If you slice AND recurse on the result, the recurrence is $T(n) = n + T(n-1)$, not $T(n) = a + T(n-1)$.

In [2]:
# EXAMPLE 1: T(n) = a + T(n-1) → Θ(n)
def count_occurrences(L, x):
    if L == []:                          # constant
        return 0
    if L[0] == x:                        # constant
        return 1 + count_occurrences(L[1:], x)  # *** slice cost is hidden here! ***
    return count_occurrences(L[1:], x)   # but typically: T(n) = a + T(n-1)

# EXAMPLE 2: T(n) = n + T(n-1) → Θ(n²) — when slice is stored in variable
def sum_list(L):
    if L == []:
        return 0
    remaining = L[1:len(L)]              # slicing costs n
    return L[0] + sum_list(remaining)    # T(n) = n + T(n-1)
# T(n) = n + (n-1) + (n-2) + ... + 1 + T(0) = n(n+1)/2  → Θ(n²)

print(count_occurrences([1,2,1,3,1], 1))
print(sum_list([1,2,3,4,5]))

3
15


In [3]:
# PRACTICE: Write T(n) for mystery (from Lecture 10 PID:35)
# Key: identify cost per level + which sub-input size is passed
def mystery(L):
    if L == []:                          # constant
        return 10
    total = 0
    for x in L:                          # n steps
        total += x * x
    left = L[0: len(L)//2]              # n/2 steps (slicing)
    return total + mystery(left)         # T(n/2)

# T(n) = c*n + T(n/2)
# Master Theorem: a=1, b=2, d=1  →  log_2(1)=0  →  d > log_b(a)  →  Θ(n)

print(mystery([3, 1, 4, 1, 5, 9]))   # try it
print(mystery([1, 2]))               # 5 + mystery([1]) = 5 + 1 + mystery([]) = 6 + 10 = 16

178
16


---
# Section 2 — Big-O, Ω, Θ

## Definitions

These measure the **growth rate** of functions — how fast T(n) grows as n → ∞.

$$T(n) \in O(f(n)) \iff \exists c > 0 \text{ such that } T(n) \le c \cdot f(n) \text{ for all large } n$$

$$T(n) \in \Omega(f(n)) \iff \exists c > 0 \text{ such that } T(n) \ge c \cdot f(n) \text{ for all large } n$$

$$T(n) \in \Theta(f(n)) \iff T(n) \in O(f(n)) \text{ AND } T(n) \in \Omega(f(n))$$

In other words: $\exists c_1, c_2 > 0$ such that $c_1 \cdot f(n) \le T(n) \le c_2 \cdot f(n)$ for all large $n$.

---

## Proof Templates

### To prove $T(n) \in O(f(n))$ — UPPER BOUND
> Replace every term in T(n) with something ≤ c·f(n).
> **Technique:** Replace lower-degree terms with the same or higher-degree terms.

**Step-by-step:**
1. Take each term of T(n) separately.
2. Upper-bound each term by a multiple of f(n).
   - If f(n) = n², then: n ≤ n², constant ≤ n²
3. Add up the constants → that's your c.
4. State: "for all n ≥ 1" (or whatever threshold you used).

### To prove $T(n) \in \Omega(f(n))$ — LOWER BOUND
> Drop terms that make T(n) smaller.
> **Technique:** Remove non-negative terms (or negative terms add them back).

**Step-by-step:**
1. Keep only the dominant term(s).
2. Drop all smaller terms (they're ≥ 0, so dropping them makes the expression smaller).
3. The remaining term = c·f(n). That's your c.

### To prove $T(n) \in \Theta(f(n))$ — TIGHT BOUND
> Do both O and Ω simultaneously:
> Find c₁ (lower) and c₂ (upper) such that $c_1 \cdot f(n) \le T(n) \le c_2 \cdot f(n)$.

## Worked Examples (Assignment 3 pattern)

---

### Show $3n^2 + 7n + 10 \in O(n^2)$

We need $c$ such that $3n^2 + 7n + 10 \le c \cdot n^2$ for all $n \ge 1$.

Upper-bound each term by $n^2$:
$$3n^2 + 7n + 10 \le 3n^2 + 7n^2 + 10n^2 = 20n^2$$

Choose $c = 20$. Then $3n^2 + 7n + 10 \le 20n^2$ for all $n \ge 1$. ✓

---

### Show $5n^2 + 3n \in \Omega(n^2)$

We need $c$ such that $5n^2 + 3n \ge c \cdot n^2$ for all $n \ge 0$.

Drop the non-negative term $3n$:
$$5n^2 + 3n \ge 5n^2$$

Choose $c = 5$. ✓

---

### Show $4n^2 + 3n + 1 \in \Theta(n^2)$

**Upper bound (O):** $4n^2 + 3n + 1 \le 4n^2 + 3n^2 + n^2 = 8n^2$ for $n \ge 1$. → $c_2 = 8$

**Lower bound (Ω):** $4n^2 + 3n + 1 \ge 4n^2$ (since $3n+1 \ge 0$). → $c_1 = 4$

$$4n^2 \le 4n^2 + 3n + 1 \le 8n^2 \text{ for all } n \ge 1 \implies 4n^2 + 3n + 1 \in \Theta(n^2) \checkmark$$

---

### Tricky case: Show $2n^2 - n \in \Omega(n^2)$

We can't just drop $-n$. For $n \ge 2$: $n \le n^2/2$, so $-n \ge -n^2/2$.
$$2n^2 - n \ge 2n^2 - \frac{n^2}{2} = \frac{3}{2}n^2$$
Choose $c = 3/2$ for all $n \ge 2$. ✓

---

## True/False Practice (Assignment 3, Q9 pattern)

| Statement | T/F | Reason |
|---|---|---|
| $n^2 + 5n \in O(n^3)$ | **True** | $n^2 \le n^3$ and $5n \le 5n^3$, so $\le 6n^3$ |
| $n^3 \in O(n^2)$ | **False** | Would require $n \le c$ for all n — impossible |
| $3n+2 \in \Theta(n)$ | **True** | $3n \le 3n+2 \le 5n$ for $n \ge 1$ |
| $n^2 + n \in \Omega(n^3)$ | **False** | $(1+1/n) \ge cn$ is impossible for any fixed $c > 0$ |
| If $T \in O(n^2)$ and $T \in \Omega(n^2)$, then $T \in \Theta(n^2)$ | **True** | This is exactly the definition of Θ |

---

## Complexity Order (important for comparisons)

$$O(1) < O(\log n) < O(n) < O(n \log n) < O(n^2) < O(n^3) < O(2^n) < O(n!)$$

---
# Section 3 — Solving Recurrences by Substitution

Use this when the Master Theorem doesn't apply (e.g., T(n) = c + T(n-1)).

## Method — 5 Steps

1. **Expand** T(n) once using the recurrence.
2. **Get T(n-1)** (or T(n/2)) by replacing n with n-1 in the original equation.
3. **Substitute** — plug T(n-1) back in.
4. **After k steps**, write the pattern: `T(n) = [expression in k] + T(n-k)` or `T(n/2^k)`.
5. **Solve for k** at the base case, substitute, simplify to get a closed form.

---

## Case 1 — T(n) = a + T(n-1), T(0) = b

*(Linear recursion with constant work — e.g., count_occurrences)*

```
T(n) = a + T(n-1)
     = a + a + T(n-2)    [replace n with n-1 in original]
     = 2a + T(n-2)
     = 3a + T(n-3)

After k steps:  T(n) = k·a + T(n-k)

Base case T(0) = b, set k = n:
  T(n) = n·a + b
```

$$T(n) = n \cdot a + b \in \Theta(n)$$

---

## Case 2 — T(n) = n + T(n-1), T(0) = b

*(Linear recursion with slicing — e.g., sum_list with L[1:] stored as variable)*

```
T(n) = n + T(n-1)
     = n + (n-1) + T(n-2)    [T(n-1) = (n-1) + T(n-2)]
     = n + (n-1) + (n-2) + T(n-3)

After k steps:  T(n) = n + (n-1) + ... + (n-k+1) + T(n-k)

Set k = n:
  T(n) = n + (n-1) + ... + 1 + T(0) = n(n+1)/2 + b
```

$$T(n) = \frac{n(n+1)}{2} + b \in \Theta(n^2)$$

---

## Case 3 — T(n) = a + T(n/2), T(1) = b

*(Binary search — one recursive call, half the input, constant work)*

```
T(n) = a + T(n/2)
     = a + a + T(n/4)    [T(n/2) = a + T(n/4)]
     = 2a + T(n/4)
     = 3a + T(n/8)

After k steps:  T(n) = k·a + T(n/2^k)

Base case T(1): set n/2^k = 1  →  k = log₂(n)
  T(n) = a·log₂(n) + b
```

$$T(n) = a \cdot \log_2(n) + b \in \Theta(\log n)$$

---

## Case 4 — T(n) = n + T(n/2), T(1) = 1

*(One recursive call on half, linear work — e.g., mystery with loop)*

```
T(n) = n + T(n/2)
     = n + n/2 + T(n/4)
     = n + n/2 + n/4 + T(n/8)

After k steps:  T(n) = n(1 + 1/2 + 1/4 + ... + 1/2^(k-1)) + T(n/2^k)

Set k = log₂(n):  geometric sum 1+1/2+... converges to 2
  T(n) ≈ 2n
```

$$T(n) \in \Theta(n)$$

---

## Solved example: T(n) = 5 + T(n-1), T(0) = 5

*(Assignment 3 style — PID:33 from Lecture 10)*

```
T(n) = 5 + T(n-1)
     = 5 + 5 + T(n-2) = 10 + T(n-2)
     = 15 + T(n-3)

After k steps:  T(n) = 5k + T(n-k)

Set k = n:  T(n) = 5n + T(0) = 5n + 5
```

$$T(n) = 5n + 5 \in \Theta(n)$$

---
# Section 4 — Master Theorem

## Statement

For recurrences of the form:
$$T(n) = n^d + a \cdot T\left(\frac{n}{b}\right)$$

where:
- $a$ = number of recursive calls
- $b$ = factor by which input size shrinks (each recursive call gets input of size $n/b$)
- $d$ = exponent of the non-recursive (combining/dividing) work: $\Theta(n^d)$

Compare $\log_b a$ and $d$:

| Case | Condition | Who dominates | Result |
|---|---|---|---|
| **Case 1** | $\log_b a > d$ | Recursive work dominates | $T(n) \in \Theta\left(n^{\log_b a}\right)$ |
| **Case 2** | $\log_b a = d$ | Balanced — neither dominates | $T(n) \in \Theta\left(n^d \log n\right)$ |
| **Case 3** | $\log_b a < d$ | Non-recursive work dominates | $T(n) \in \Theta\left(n^d\right)$ |

> **Memory aid:**  
> - $\log_b a$ tells you how fast the recursion "fans out" (more subproblems → bigger recursive cost).  
> - $d$ tells you how expensive the non-recursive work is at each level.  
> - Whichever is bigger wins. If tied, multiply by $\log n$.

---

## ⚠️ IMPORTANT: When Master Theorem does NOT apply

The Master Theorem **only** applies when the input shrinks by a **constant factor** ($n/b$).

It does **NOT** apply to:
- $T(n) = c + T(n-1)$ — input shrinks by 1, not a factor → **use substitution**
- $T(n) = n + T(n-1)$ — same issue
- $T(n) = T(n-1) + T(n-2)$ — two calls, both shrink by 1 → **use Ω analysis / substitution**

---

## Quick Reference Table

| Recurrence | a | b | d | $\log_b a$ | Case | Result |
|---|---|---|---|---|---|---|
| $T(n) = 1 + T(n/2)$ | 1 | 2 | 0 | 0 | **2** ($=$) | $\Theta(\log n)$ |
| $T(n) = n + T(n/2)$ | 1 | 2 | 1 | 0 | **3** ($<$) | $\Theta(n)$ |
| $T(n) = n + 2T(n/2)$ | 2 | 2 | 1 | 1 | **2** ($=$) | $\Theta(n \log n)$ |
| $T(n) = n + 4T(n/2)$ | 4 | 2 | 1 | 2 | **1** ($>$) | $\Theta(n^2)$ |
| $T(n) = n + 3T(n/2)$ | 3 | 2 | 1 | $\approx 1.58$ | **1** ($>$) | $\Theta(n^{\log_2 3})$ |
| $T(n) = n^2 + 2T(n/2)$ | 2 | 2 | 2 | 1 | **3** ($<$) | $\Theta(n^2)$ |
| $T(n) = 5n^3 + 8T(n/2)$ | 8 | 2 | 3 | 3 | **2** ($=$) | $\Theta(n^3 \log n)$ |
| $T(n) = 4 + T(n/4)$ | 1 | 4 | 0 | 0 | **2** ($=$) | $\Theta(\log n)$ |
| $T(n) = 1 + 2T(n/2)$ | 2 | 2 | 0 | 1 | **1** ($>$) | $\Theta(n)$ |

## Worked Examples — Applying the 3 Cases

### Example 1: $T(n) = n + 2T(n/2)$ ← merge sort
- $a=2$, $b=2$, $d=1$
- $\log_b a = \log_2 2 = 1$
- $\log_b a = d = 1$ → **Case 2** (balanced)

$$T(n) \in \Theta(n^1 \log n) = \Theta(n \log n)$$

---

### Example 2: $T(n) = n + 4T(n/2)$ ← naive integer multiplication
- $a=4$, $b=2$, $d=1$
- $\log_b a = \log_2 4 = 2$
- $\log_b a = 2 > 1 = d$ → **Case 1** (recursive dominates)

$$T(n) \in \Theta(n^{\log_2 4}) = \Theta(n^2)$$

---

### Example 3: $T(n) = n^2 + 2T(n/2)$ ← quadratic combine step
- $a=2$, $b=2$, $d=2$
- $\log_b a = \log_2 2 = 1$
- $\log_b a = 1 < 2 = d$ → **Case 3** (non-recursive dominates)

$$T(n) \in \Theta(n^2)$$

---

### Example 4: $T(n) = n^2 + 4T(n/2)$
- $a=4$, $b=2$, $d=2$
- $\log_b a = \log_2 4 = 2$
- $\log_b a = d = 2$ → **Case 2** (balanced)

$$T(n) \in \Theta(n^2 \log n)$$

---

### Example 5: $T(n) = n + 3T(n/2)$ ← Karatsuba
- $a=3$, $b=2$, $d=1$
- $\log_b a = \log_2 3 \approx 1.585$
- $\log_b a \approx 1.585 > 1 = d$ → **Case 1** (recursive dominates)

$$T(n) \in \Theta(n^{\log_2 3}) \approx \Theta(n^{1.585})$$

---

### Example 6: $T(n) = n^3 + 2T(n/4)$
- $a=2$, $b=4$, $d=3$
- $\log_b a = \log_4 2 = 0.5$
- $\log_b a = 0.5 < 3 = d$ → **Case 3** (non-recursive dominates)

$$T(n) \in \Theta(n^3)$$

---

### Useful log facts
$$\log_b a = d \iff b^d = a$$
$$\log_2 1 = 0,\quad \log_2 2 = 1,\quad \log_2 4 = 2,\quad \log_2 8 = 3,\quad \log_4 2 = 0.5$$
$$\log_2 3 \approx 1.585,\quad \log_3 3 = 1,\quad \log_3 9 = 2$$

---
# Section 5 — Divide and Conquer

## Strategy in English (exam template)

> *"To solve a problem of size n: **(1) divide** the input into two halves. **(2) Recursively solve** each half. **(3) Combine** the two results to get the answer for the whole input."*

## When D&C helps
- Subproblems are **independent** (don't share work) — contrast with DP
- The combine step is **cheap** (ideally O(n) or O(1))
- The problem has **optimal substructure**: the optimal solution to the whole can be built from optimal solutions to parts

## When D&C doesn't help
- You're just halving the input but still doing O(n) total work per level with no benefit
- Subproblems overlap — use DP instead

In [4]:
# Binary Search — T(n) = 1 + T(n/2) → Θ(log n)
# Strategy: check the middle; recurse on left or right half only
def binary_search(L, x, lo, hi):
    if lo > hi:
        return False
    mid = (lo + hi) // 2
    if x == L[mid]:
        return True
    elif x < L[mid]:
        return binary_search(L, x, lo, mid - 1)  # T(n/2)
    else:
        return binary_search(L, x, mid + 1, hi)  # T(n/2)
# Non-recursive work: O(1) = n^0  →  a=1, b=2, d=0  →  d=log_b(a)=0  →  Θ(log n)

L = [1, 3, 5, 7, 9, 11]
print(binary_search(L, 7, 0, len(L)-1))   # True
print(binary_search(L, 4, 0, len(L)-1))   # False

True
False


In [5]:
# Fast Exponentiation — T(n) = 1 + T(n/2) → Θ(log n)
# Strategy: x^n = (x^(n/2))^2 if n even; x^n = x * x^(n-1) if n odd
def power(base, exp):
    if exp == 0:
        return 1
    if exp % 2 == 0:
        half = power(base, exp // 2)  # T(n/2)
        return half * half             # O(1) combine
    return base * power(base, exp - 1) # T(n-1) — only for odd
# T(exp) = Θ(log exp)  vs naive Θ(exp)

print(power(2, 10))   # 1024
print(power(3, 5))    # 243

1024
243


In [6]:
# Merge Sort — T(n) = n + 2T(n/2) → Θ(n log n)
# Strategy: split in half, sort each half, merge the two sorted halves
def merge(left, right):
    result = []
    i = j = 0
    while i < len(left) and j < len(right):
        if left[i] <= right[j]:
            result.append(left[i]); i += 1
        else:
            result.append(right[j]); j += 1
    result.extend(left[i:])
    result.extend(right[j:])
    return result

def merge_sort(L):
    if len(L) <= 1:
        return L
    mid = len(L) // 2
    left  = merge_sort(L[:mid])   # T(n/2)
    right = merge_sort(L[mid:])   # T(n/2)
    return merge(left, right)     # Θ(n) to merge
# T(n) = n + 2T(n/2)  →  a=2, b=2, d=1  →  d=log_2(2)=1  →  Θ(n log n)

print(merge_sort([5, 2, 8, 1, 9, 3]))  # [1, 2, 3, 5, 8, 9]

[1, 2, 3, 5, 8, 9]


In [7]:
# Count D&C (Assignment 4) — T(n) = n + 2T(n/2)
# Strategy: split in half, count in each half, add results
def count_dc(L, x):
    if len(L) == 0: return 0
    if len(L) == 1: return 1 if L[0] == x else 0
    mid = len(L) // 2
    return count_dc(L[:mid], x) + count_dc(L[mid:], x)
# T(n) = n + 2T(n/2)  →  Θ(n log n)  [worse than simple O(n) scan!]
# Lesson: D&C doesn't always help. Here it HURTS.

# Min-Max D&C (Assignment 4)
def min_max(L):
    if len(L) == 1:
        return [L[0], L[0]]
    mid = len(L) // 2
    left_mm  = min_max(L[:mid])
    right_mm = min_max(L[mid:])
    return [min(left_mm[0], right_mm[0]), max(left_mm[1], right_mm[1])]

print(count_dc([1,2,1,3,1,2], 1))  # 3
print(min_max([5, 2, 8, 1, 9, 3])) # [1, 9]

3
[1, 9]


## Karatsuba Multiplication — the key insight (Assignment 6)

**Problem:** Multiply two n-digit numbers.

**Naive D&C:** Split each number into two halves: $x = a \cdot 10^{n/2} + b$, $y = c \cdot 10^{n/2} + d$
$$xy = ac \cdot 10^n + (ad + bc) \cdot 10^{n/2} + bd$$
This needs **4 multiplications**: $ac$, $ad$, $bc$, $bd$ → $T(n) = n + 4T(n/2)$ → $\Theta(n^2)$

**Karatsuba's trick:** Compute only 3 multiplications:
1. $ac$
2. $bd$
3. $(a+b)(c+d) = ac + ad + bc + bd$

Then $ad + bc = (a+b)(c+d) - ac - bd$ — **derived without extra multiplication!**

→ $T(n) = n + 3T(n/2)$ → $a=3, b=2, d=1$, $\log_2 3 \approx 1.585 > 1 = d$ → $\Theta(n^{\log_2 3}) \approx \Theta(n^{1.585})$

**Key: only $a$ changed (4 → 3). $b$ and $d$ stayed the same.**

## "Explain in English" Template for D&C

When asked to explain a D&C strategy:

> *"**Base case:** If the input has size 0 or 1, return [trivial answer].  
> **Divide:** Split the input into two halves.  
> **Conquer:** Recursively solve each half.  
> **Combine:** [Specific operation — merge, compare, add, etc.] the two results to get the answer for the whole input."*

**Example for merge sort:**
> *"If the list has 0 or 1 elements, it is already sorted. Otherwise, split the list into two halves. Recursively sort each half. Then merge the two sorted halves by repeatedly taking the smaller front element from either half."*

**Example for min-max:**
> *"If the list has 1 element, it is both the min and max. Otherwise, split in half, find [min, max] of each half recursively. The overall min is the smaller of the two mins, and the overall max is the larger of the two maxes."*

---
# Section 6 — Dynamic Programming (Memoization)

## When to Use DP

Use DP when you have a recursive solution where the **same subproblem is computed multiple times** (overlapping subproblems).

**Diagnosis:** Draw the recursion tree. If you see the same call appearing at multiple branches → overlapping subproblems → use memoization.

**Key insight:** Cache (store) the result of each call. If you need it again, just look it up instead of recomputing.

---

## Memoization Template

```python
Cache = {}

def f(input):
    # Step 1: Look up first — return immediately if already computed
    if input in Cache:
        return Cache[input]
    
    # Step 2: Base case
    if input == base_case_value:
        Cache[input] = base_case_result
        return Cache[input]
    
    # Step 3: Recurse
    result = f(smaller_input_1) + f(smaller_input_2)  # or other combination
    
    # Step 4: Store before returning
    Cache[input] = result
    return Cache[input]
```

**The 5 questions to answer when adding caching:**
1. What do we store? → the output/return value
2. When do we store? → just before returning
3. Where do we store? → a dictionary (Cache)
4. How do we store? → `Cache[input] = output`
5. When do we look up? → **first thing**, before any computation

In [8]:
# EXAMPLE 1: Staircase climbing — climb n steps, each step is 1 or 2 stairs

# WITHOUT cache — exponential T(n) = T(n-1) + T(n-2)  →  Θ(1.62^n)
def climb_slow(n):
    if n == 0 or n == 1:
        return 1
    return climb_slow(n-1) + climb_slow(n-2)  # climb(38) computed TWICE from climb(40)

# WITH cache — Θ(n)
Cache_climb = {}
def climb(n):
    if n in Cache_climb:                        # look up first
        return Cache_climb[n]
    if n == 0 or n == 1:
        Cache_climb[n] = 1
        return 1
    output = climb(n-1) + climb(n-2)
    Cache_climb[n] = output                     # store before returning
    return output

print(climb(10))   # 89
print(climb(50))   # large number, computed instantly

89
20365011074


In [9]:
# EXAMPLE 2: Making change — can we make amount A using coins?
# Recurrence: is_changeable(A) = any(is_changeable(A-c) for c in coins)

Cache_change = {}
def is_changeable(A, coins):
    if A in Cache_change:
        return Cache_change[A]
    if A == 0:
        Cache_change[A] = True
        return True
    if A < 0:
        Cache_change[A] = False
        return False
    for c in coins:
        if is_changeable(A - c, coins):
            Cache_change[A] = True
            return True
    Cache_change[A] = False
    return False

print(is_changeable(13, [5, 6, 3]))   # True  (5+5+3)
print(is_changeable(7,  [5, 6, 3]))   # False
print(is_changeable(11, [5, 6, 3]))   # True  (5+6)

True
False
True


In [10]:
# EXAMPLE 3: Count paths on grid — move only left or down from (x,y) to (1,1)
# Recurrence: count_paths(x,y) = count_paths(x-1,y) + count_paths(x,y-1)

Cache_paths = {}
def count_paths(x, y):
    if (x, y) in Cache_paths:
        return Cache_paths[x, y]
    if x <= 1 or y <= 1:
        Cache_paths[x, y] = 1
        return 1
    Cache_paths[x, y] = count_paths(x-1, y) + count_paths(x, y-1)
    return Cache_paths[x, y]
# Without cache: Θ(2^n). With cache: each (x,y) computed once → Θ(n²)

print(count_paths(3, 3))   # 6
print(count_paths(5, 5))   # 70

6
70


In [11]:
# EXAMPLE 4: Rod Cutting (Assignment 7) — maximize revenue r(n)
# Recurrence: r(n) = max over i in {1..n} of { p[i] + r(n-i) },  r(0) = 0

P = [0, 1, 5, 8, 9, 10, 17, 17, 20]  # P[i] = price of piece of length i

# WITHOUT cache — Θ(2^n) because r(2) computed from r(4), r(3), etc.
def rod_cut_dp(p, n):
    if n == 0: return 0
    best = p[n]               # option: don't cut at all
    for i in range(1, n):     # try every first-piece length
        candidate = p[i] + rod_cut_dp(p, n - i)
        if candidate > best:
            best = candidate
    return best

# WITH cache — Θ(n²): compute r(0), r(1), ..., r(n) each once, each taking Θ(n)
def rod_cut_memo(p, n):
    cache = {}
    def helper(m):
        if m in cache: return cache[m]
        if m == 0:
            cache[m] = 0
            return 0
        best = p[m]
        for i in range(1, m):
            candidate = p[i] + helper(m - i)
            if candidate > best: best = candidate
        cache[m] = best
        return best
    return helper(n)

print(rod_cut_memo(P, 4))  # 10  (cut: 2+2)
print(rod_cut_memo(P, 5))  # 13  (cut: 3+2)
print(rod_cut_memo(P, 8))  # 22  (cut: 6+2)

10
13
22


## DP — Running Time Summary

| Problem | Without cache | With cache |
|---|---|---|
| `climb(n)` | $\Theta(\phi^n) \approx \Theta(1.62^n)$ | $\Theta(n)$ |
| `is_changeable(A, coins)` | exponential | $\Theta(A \cdot |coins|)$ |
| `count_paths(n,n)` | $\Omega(2^n)$ | $\Theta(n^2)$ |
| `rod_cut(p, n)` | $\Theta(2^n)$ | $\Theta(n^2)$ |

---

## "Explain in English" Template for DP

> *"We identify that this problem has overlapping subproblems: [specific example of repeated computation]. We fix this by caching: before computing f(x), we check if the answer is already stored. If yes, return it immediately. If no, compute it recursively, store the result, then return it. This ensures each subproblem is computed exactly once."*

---
# Section 7 — Backtracking

## When to Use Backtracking

Use backtracking when you need to **enumerate all possible solutions** (or find one/the best) and the solution can be built one piece at a time.

## The General Template (exact from Lecture 15)

```python
def backtrack(solution, 
              possibilities, 
              i = 0, 
              is_valid=lambda solution: True,
              display_solution=lambda s: s):
    if i == len(solution):
        if is_valid(solution):
            print(display_solution(solution))
    else:
        for possibility in possibilities(solution, i):
            solution[i] = possibility
            backtrack(solution, possibilities, i+1, is_valid, display_solution)
```

## Parameter Guide

| Parameter | Role | Example |
|---|---|---|
| `solution` | List of length n; represents one complete solution | `[None]*4` |
| `i` | Current position being filled (0 to n-1) | start with `i=0` |
| `possibilities(solution, i)` | Returns valid choices for position i given current partial solution | `lambda sol, i: [0,1]` |
| `is_valid(solution)` | Final check on complete solution | `lambda s: s.count(1)==2` |
| `display_solution(solution)` | How to format/display the solution | `lambda s: ''.join(map(str,s))` |

## Two Types of Solutions

### Type 1: Sequences — `solution[i]` holds the actual item
```python
solution = [None]*4          # 4-digit binary number
possibilities = lambda sol, i: [0, 1]
```

### Type 2: Indicators/Subsets — `solution[i]` is True/False
```python
items = ['apple', 'orange', 'banana']
solution = [None]*3          # solution[i]=True means items[i] is in subset
possibilities = lambda sol, i: [True, False]
display = lambda sol: {items[i] for i in range(len(sol)) if sol[i]}
```

## Tree Pruning — move constraints into `possibilities`

**Without pruning** (check at end — slow): try all possibilities, filter at the end with `is_valid`.

**With pruning** (check early — fast): make `possibilities` return only valid choices for position i based on the current partial solution. This avoids whole branches that can never lead to valid solutions.

In [12]:
# THE GENERAL TEMPLATE
def backtrack(solution, 
              possibilities, 
              i = 0, 
              is_valid=lambda solution: True,
              display_solution=lambda s: s):
    if i == len(solution):
        if is_valid(solution):
            print(display_solution(solution))
    else:
        for possibility in possibilities(solution, i):
            solution[i] = possibility
            backtrack(solution, possibilities, i+1, is_valid, display_solution)

In [13]:
# EXAMPLE 1: All 3-digit binary numbers
backtrack(
    solution=[None]*3,
    possibilities=lambda sol, i: [0, 1]
)
# Prints: [0,0,0], [0,0,1], [0,1,0], ..., [1,1,1]  (2^3 = 8 solutions)

[0, 0, 0]
[0, 0, 1]
[0, 1, 0]
[0, 1, 1]
[1, 0, 0]
[1, 0, 1]
[1, 1, 0]
[1, 1, 1]


In [14]:
# EXAMPLE 2: 4-digit binary numbers with NO consecutive 1s
# PRUNE: if previous digit was 1, current digit can only be 0
def no_consecutive_1s(sol, i):
    if i == 0 or sol[i-1] == 0:
        return [0, 1]   # both allowed
    else:
        return [0]      # previous was 1, so only 0 allowed

backtrack(
    solution=[None]*4,
    possibilities=no_consecutive_1s,
    display_solution=lambda s: ''.join(map(str, s))
)

0000
0001
0010
0100
0101
1000
1001
1010


In [15]:
# EXAMPLE 3: All subsets of ['apple', 'orange', 'banana']
fruits = ['apple', 'orange', 'banana']

backtrack(
    solution=[None]*3,
    possibilities=lambda sol, i: [True, False],
    display_solution=lambda sol: {fruits[i] for i in range(len(sol)) if sol[i]}
)

{'orange', 'apple', 'banana'}
{'orange', 'apple'}
{'banana', 'apple'}
{'apple'}
{'orange', 'banana'}
{'orange'}
{'banana'}
set()


In [16]:
# EXAMPLE 4: Rod Cutting with Backtracking (Assignment 7 pattern)
#
# Represent cuts as an indicator list of length n-1:
#   solution[i] = True  means "cut at gap i" (between positions i+1 and i+2)
#   solution[i] = False means "don't cut at gap i"
# A rod of length n has n-1 gaps.

P = [0, 1, 5, 8, 9, 10, 17, 17, 20]

def rod_cut_backtrack(p, n):
    if n == 0: return 0
    best = [0]
    solution = [None] * (n - 1)

    def visit(sol):
        # Decode indicator list → piece lengths
        pieces, start = [], 0
        for i in range(len(sol)):
            if sol[i]:
                pieces.append(i + 1 - start)
                start = i + 1
        pieces.append(n - start)
        # Compute total revenue
        total = 0
        for piece in pieces:
            total += p[piece]
        if total > best[0]:
            best[0] = total
        return (pieces, total)

    backtrack(solution, lambda sol, i: [False, True], display_solution=visit)
    return best[0]

print(rod_cut_backtrack(P, 4))  # 10 (cut: 2+2)
print(rod_cut_backtrack(P, 5))  # 13 (cut: 3+2)

([4], 9)
([3, 1], 9)
([2, 2], 10)
([2, 1, 1], 7)
([1, 3], 9)
([1, 2, 1], 7)
([1, 1, 2], 7)
([1, 1, 1, 1], 4)
10
([5], 10)
([4, 1], 10)
([3, 2], 13)
([3, 1, 1], 10)
([2, 3], 13)
([2, 2, 1], 11)
([2, 1, 2], 11)
([2, 1, 1, 1], 8)
([1, 4], 10)
([1, 3, 1], 10)
([1, 2, 2], 11)
([1, 2, 1, 1], 8)
([1, 1, 3], 10)
([1, 1, 2, 1], 8)
([1, 1, 1, 2], 8)
([1, 1, 1, 1, 1], 5)
13


## "Explain in English" Template for Backtracking

> *"We represent each solution as a list of length n. We fill in positions 0 to n-1 one at a time. At each position i, we try every possible value (from the possibilities function). If the partial solution so far cannot lead to a valid complete solution, we skip that branch (pruning). When all positions are filled (i = n), we check if the solution is valid and, if so, process it (print, update best, etc.)."*

**For rod cutting:**
> *"We represent a cutting as a True/False indicator list of length n-1, where True at position i means we cut at gap i. We try both True and False for each gap in order. When all gaps are decided, we decode the indicator list into piece lengths, compute the total revenue, and update the best if this revenue is higher."*

---
# Quick Reference Cheat Sheet

---

## Complexity Order
$$O(1) < O(\log n) < O(n) < O(n \log n) < O(n^2) < O(n^3) < O(2^n) < O(n!)$$

---

## Common T(n) → Complexity

| T(n) | Method | Complexity |
|---|---|---|
| $a + T(n-1)$ | Substitution | $\Theta(n)$ |
| $n + T(n-1)$ | Substitution | $\Theta(n^2)$ |
| $a + T(n/2)$ | Master Case 2 | $\Theta(\log n)$ |
| $n + T(n/2)$ | Master Case 3 | $\Theta(n)$ |
| $n + 2T(n/2)$ | Master Case 2 | $\Theta(n \log n)$ |
| $n + 4T(n/2)$ | Master Case 1 | $\Theta(n^2)$ |
| $n + 3T(n/2)$ | Master Case 1 | $\Theta(n^{\log_2 3})$ |
| $1 + 2T(n/2)$ | Master Case 1 | $\Theta(n)$ |
| $T(n-1) + T(n-2)$ | Substitution | $\Theta(\phi^n)$ |
| $a + b \cdot n^2$ | Direct | $\Theta(n^2)$ |
| $a + b \cdot n(n-1)/2$ | Direct | $\Theta(n^2)$ |

---

## Master Theorem — 3-Case Summary

For $T(n) = n^d + a \cdot T(n/b)$, compute $\log_b a$ then:

| Case | Condition | Who wins | Result |
|---|---|---|---|
| **1** | $\log_b a > d$ | Recursive work | $\Theta(n^{\log_b a})$ |
| **2** | $\log_b a = d$ | Neither (balanced) | $\Theta(n^d \log n)$ |
| **3** | $\log_b a < d$ | Non-recursive work | $\Theta(n^d)$ |

⚠️ Only for recurrences with $n/b$ (not $n-1$)!

---

## When to Use Which Strategy

| Strategy | Use when... | Running time |
|---|---|---|
| **Divide & Conquer** | Subproblems are independent; split gives efficiency | Typically $\Theta(n \log n)$ or better |
| **Dynamic Programming** | Same subproblems appear multiple times; cache results | Depends — often polynomial |
| **Backtracking** | Need all solutions; build one choice at a time; prune invalid branches | Worst case exponential; pruning helps |

---

## Substitution Method — Steps

1. Expand T(n) once.
2. Replace n with n-1 (or n/2) in original → get T(n-1).
3. Substitute. Repeat until pattern is visible.
4. Write: "After k steps: T(n) = [pattern(k)] + T(n-k or n/2^k)".
5. Set k at base case (n-k=0 or n/2^k=1). Solve for k. Substitute.

---

## Big-O / Ω / Θ Proof — Summary

**O (upper bound):** Replace each term with something ≤ c·f(n). Add constants → c. State n₀.

**Ω (lower bound):** Drop non-negative terms to get something ≥ c·f(n).

**Θ (tight bound):** Do both. Find c₁ ≤ T(n)/f(n) ≤ c₂.

---

## Backtracking — Quick Checklist

1. What does `solution` represent? (sequence or indicator list?)
2. What are valid choices for `solution[i]`? (defines `possibilities`)
3. Can I prune early? (move constraints into `possibilities`)
4. What makes a complete solution valid? (defines `is_valid`)
5. How to display the solution? (defines `display_solution`)

---

## Memoization — Quick Checklist

1. Does the recursion compute the same input twice? → Need caching.
2. Add `Cache = {}`.
3. **First line** of function: `if input in Cache: return Cache[input]`.
4. **Before every return**: `Cache[input] = result`.
5. Key is the input; value is the output.